In [1]:
from pathlib import Path
import sys

import pandas as pd

# Detect project root
PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

# Add project root to Python path
sys.path.append(str(PROJECT_ROOT))

from src.data_loader import load_all_raw_data

In [2]:
dataframes = load_all_raw_data()

list(dataframes.keys())

All expected raw CSV files are available.


['olist_customers_dataset',
 'olist_geolocation_dataset',
 'olist_order_items_dataset',
 'olist_order_payments_dataset',
 'olist_order_reviews_dataset',
 'olist_orders_dataset',
 'olist_products_dataset',
 'olist_sellers_dataset',
 'product_category_name_translation']

In [7]:
table_summary = []

for table_name, df in dataframes.items():
    table_summary.append(
        {
            "table_name": table_name,
            "row_count": df.shape[0],
            "column_count": df.shape[1],
        }
    )

table_summary_df = pd.DataFrame(table_summary)
table_summary_df.sort_values("row_count", ascending=False)

,table_name,row_count,column_count
1,olist_geolocation_dataset,1000163,5
2,olist_order_items_dataset,112650,7
3,olist_order_payments_dataset,103886,5
0,olist_customers_dataset,99441,5
5,olist_orders_dataset,99441,8
4,olist_order_reviews_dataset,99224,7
6,olist_products_dataset,32951,9
7,olist_sellers_dataset,3095,4
8,product_category_name_translation,71,2


In [9]:
columns_summary = []

for table_name, df in dataframes.items():
    for column in df.columns:
        columns_summary.append(
            {
                "table_name": table_name,
                "column_name": column,
                "data_type": str(df[column].dtype),
                "non_null_count": df[column].notna().sum(),
                "missing_count": df[column].isna().sum(),
                "unique_count": df[column].nunique(),
            }
        )

columns_summary_df = pd.DataFrame(columns_summary)

output_path = PROJECT_ROOT / "outputs" / "table_columns_summary.csv"
columns_summary_df.to_csv(output_path, index=False)

columns_summary_df.head(20)

,table_name,column_name,data_type,non_null_count,missing_count,unique_count
0,olist_customers_dataset,customer_id,str,99441,0,99441
1,olist_customers_dataset,customer_unique_id,str,99441,0,96096
2,olist_customers_dataset,customer_zip_code_prefix,int64,99441,0,14994
3,olist_customers_dataset,customer_city,str,99441,0,4119
4,olist_customers_dataset,customer_state,str,99441,0,27
5,olist_geolocation_dataset,geolocation_zip_code_prefix,int64,1000163,0,19015
6,olist_geolocation_dataset,geolocation_lat,float64,1000163,0,717360
7,olist_geolocation_dataset,geolocation_lng,float64,1000163,0,717613
8,olist_geolocation_dataset,geolocation_city,str,1000163,0,8011
9,olist_geolocation_dataset,geolocation_state,str,1000163,0,27


In [10]:
orders = dataframes["olist_orders_dataset"]
order_items = dataframes["olist_order_items_dataset"]
customers = dataframes["olist_customers_dataset"]
products = dataframes["olist_products_dataset"]
payments = dataframes["olist_order_payments_dataset"]
reviews = dataframes["olist_order_reviews_dataset"]
sellers = dataframes["olist_sellers_dataset"]
category_translation = dataframes["product_category_name_translation"]

orders.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00


In [11]:
orders_dates = orders.copy()

date_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
]

for column in date_columns:
    orders_dates[column] = pd.to_datetime(orders_dates[column], errors="coerce")

orders_dates[date_columns].agg(["min", "max"])

,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
min,2016-09-04 21:15:19,2016-09-15 12:16:38,2016-10-08 10:34:01,2016-10-11 13:46:32,2016-09-30
max,2018-10-17 17:30:18,2018-09-03 17:40:06,2018-09-11 19:48:28,2018-10-17 13:22:46,2018-11-12


In [12]:
key_checks = [
    ("olist_orders_dataset", ["order_id"]),
    ("olist_customers_dataset", ["customer_id"]),
    ("olist_products_dataset", ["product_id"]),
    ("olist_sellers_dataset", ["seller_id"]),
    ("product_category_name_translation", ["product_category_name"]),
    ("olist_order_items_dataset", ["order_id", "order_item_id"]),
]

primary_key_results = []

for table_name, key_columns in key_checks:
    df = dataframes[table_name]
    duplicate_count = df.duplicated(subset=key_columns).sum()

    primary_key_results.append(
        {
            "table_name": table_name,
            "key_columns": ", ".join(key_columns),
            "row_count": len(df),
            "duplicate_key_count": duplicate_count,
            "is_unique": duplicate_count == 0,
        }
    )

primary_key_results_df = pd.DataFrame(primary_key_results)
primary_key_results_df

,table_name,key_columns,row_count,duplicate_key_count,is_unique
0,olist_orders_dataset,order_id,99441,0,True
1,olist_customers_dataset,customer_id,99441,0,True
2,olist_products_dataset,product_id,32951,0,True
3,olist_sellers_dataset,seller_id,3095,0,True
4,product_category_name_translation,product_category_name,71,0,True
5,olist_order_items_dataset,"order_id, order_item_id",112650,0,True


In [13]:
def check_foreign_key(child_df, child_key, parent_df, parent_key):
    child_unique = child_df[[child_key]].drop_duplicates()
    matched = child_unique[child_key].isin(parent_df[parent_key])

    total_unique_child_keys = len(child_unique)
    unmatched_count = (~matched).sum()
    matched_percentage = round((matched.sum() / total_unique_child_keys) * 100, 2)

    return total_unique_child_keys, unmatched_count, matched_percentage


relationship_checks = []

relationships = [
    {
        "relationship": "orders.customer_id → customers.customer_id",
        "child_table": "olist_orders_dataset",
        "child_key": "customer_id",
        "parent_table": "olist_customers_dataset",
        "parent_key": "customer_id",
    },
    {
        "relationship": "order_items.order_id → orders.order_id",
        "child_table": "olist_order_items_dataset",
        "child_key": "order_id",
        "parent_table": "olist_orders_dataset",
        "parent_key": "order_id",
    },
    {
        "relationship": "payments.order_id → orders.order_id",
        "child_table": "olist_order_payments_dataset",
        "child_key": "order_id",
        "parent_table": "olist_orders_dataset",
        "parent_key": "order_id",
    },
    {
        "relationship": "reviews.order_id → orders.order_id",
        "child_table": "olist_order_reviews_dataset",
        "child_key": "order_id",
        "parent_table": "olist_orders_dataset",
        "parent_key": "order_id",
    },
    {
        "relationship": "order_items.product_id → products.product_id",
        "child_table": "olist_order_items_dataset",
        "child_key": "product_id",
        "parent_table": "olist_products_dataset",
        "parent_key": "product_id",
    },
    {
        "relationship": "order_items.seller_id → sellers.seller_id",
        "child_table": "olist_order_items_dataset",
        "child_key": "seller_id",
        "parent_table": "olist_sellers_dataset",
        "parent_key": "seller_id",
    },
    {
        "relationship": "products.product_category_name → category_translation.product_category_name",
        "child_table": "olist_products_dataset",
        "child_key": "product_category_name",
        "parent_table": "product_category_name_translation",
        "parent_key": "product_category_name",
    },
]

for item in relationships:
    total_keys, unmatched_count, matched_percentage = check_foreign_key(
        dataframes[item["child_table"]],
        item["child_key"],
        dataframes[item["parent_table"]],
        item["parent_key"],
    )

    relationship_checks.append(
        {
            "relationship": item["relationship"],
            "total_unique_child_keys": total_keys,
            "unmatched_count": unmatched_count,
            "matched_percentage": matched_percentage,
        }
    )

relationship_checks_df = pd.DataFrame(relationship_checks)

output_path = PROJECT_ROOT / "outputs" / "key_relationship_checks.csv"
relationship_checks_df.to_csv(output_path, index=False)

relationship_checks_df

,relationship,total_unique_child_keys,unmatched_count,matched_percentage
0,orders.customer_id → customers.customer_id,99441,0,100.00
1,order_items.order_id → orders.order_id,98666,0,100.00
2,payments.order_id → orders.order_id,99440,0,100.00
3,reviews.order_id → orders.order_id,98673,0,100.00
4,order_items.product_id → products.product_id,32951,0,100.00
5,order_items.seller_id → sellers.seller_id,3095,0,100.00
6,products.product_category_name → category_tran...,74,3,95.95


In [17]:
missing_values_path = PROJECT_ROOT / "outputs" / "missing_values_summary.csv"

missing_values_summary = pd.read_csv(missing_values_path)

missing_values_summary.query("missing_count > 0").sort_values(
    by="missing_percentage",
    ascending=False
)

,table_name,column_name,missing_count,missing_percentage,data_type
25,olist_order_reviews_dataset,review_comment_title,87656,88.34,str
26,olist_order_reviews_dataset,review_comment_message,58247,58.70,str
35,olist_orders_dataset,order_delivered_customer_date,2965,2.98,str
39,olist_products_dataset,product_name_lenght,610,1.85,float64
38,olist_products_dataset,product_category_name,610,1.85,str
40,olist_products_dataset,product_description_lenght,610,1.85,float64
41,olist_products_dataset,product_photos_qty,610,1.85,float64
34,olist_orders_dataset,order_delivered_carrier_date,1783,1.79,str
33,olist_orders_dataset,order_approved_at,160,0.16,str
42,olist_products_dataset,product_weight_g,2,0.01,float64


In [21]:
duplicate_rows_path = PROJECT_ROOT / "outputs" / "duplicate_rows_summary.csv"

duplicate_rows_summary = pd.read_csv(duplicate_rows_path)

duplicate_rows_summary.sort_values(
    by="duplicate_rows",
    ascending=False
)

,table_name,duplicate_rows,duplicate_percentage
1,olist_geolocation_dataset,261831,26.18
0,olist_customers_dataset,0,0.00
2,olist_order_items_dataset,0,0.00
3,olist_order_payments_dataset,0,0.00
4,olist_order_reviews_dataset,0,0.00
5,olist_orders_dataset,0,0.00
6,olist_products_dataset,0,0.00
7,olist_sellers_dataset,0,0.00
8,product_category_name_translation,0,0.00


In [16]:
report_path = PROJECT_ROOT / "reports" / "initial_data_understanding.md"

total_tables = len(dataframes)
total_rows = sum(df.shape[0] for df in dataframes.values())

largest_table = table_summary_df.sort_values(
    by="row_count",
    ascending=False
).iloc[0]

report_content = f"""# Initial Data Understanding

## Project

Olist E-Commerce BI Analytics

## Dataset Overview

- Total tables loaded: {total_tables}
- Total raw rows across all tables: {total_rows:,}
- Largest table: {largest_table["table_name"]}
- Largest table row count: {largest_table["row_count"]:,}

## Main Tables

The main analytical tables are:

- `olist_orders_dataset`
- `olist_order_items_dataset`
- `olist_order_payments_dataset`
- `olist_order_reviews_dataset`
- `olist_customers_dataset`
- `olist_products_dataset`
- `olist_sellers_dataset`
- `product_category_name_translation`

## Initial Notes

- `olist_orders_dataset` will be the main transaction table.
- `olist_order_items_dataset` will be used to calculate revenue from product price and freight value.
- `olist_customers_dataset` will support customer location analysis.
- `olist_products_dataset` and `product_category_name_translation` will support product category analysis.
- `olist_order_reviews_dataset` will support customer satisfaction analysis.
- `olist_order_payments_dataset` will support payment behavior analysis.

## Next Step

The next step is to clean and transform raw data before loading it into SQLite.
"""

report_path.write_text(report_content, encoding="utf-8")

print(f"Report saved to: {report_path}")

Report saved to: c:\olist-ecommerce-bi-analytics\reports\initial_data_understanding.md
